# Merging DataFrames

In [ ]:
import pandas as pd

## Our Datasets
- Our datasets are located in multiple files in the `library/` folder.
- The `books.csv` file stores the library's book catalog.
- The `members.csv` file stores the library's registered members.
- The `checkouts_jan` and `checkouts_feb` files stores borrowing records from January and February.

In [ ]:
book = pd.read_csv("library/books.csv")
book.head()

In [ ]:
members = pd.read_csv("library/members.csv", parse_dates=["Signup Date"])
members.head()

In [ ]:
jan = pd.read_csv("library/checkouts_jan.csv")
feb = pd.read_csv("library/checkouts_feb.csv")
feb.head()

## The pd.concat Function I
- The `concat` function concatenates one `DataFrame` to the end of another.
- The original index labels will be kept by default. Set `ignore_index` to `True` to generate a new index.

In [ ]:
pd.concat([jan, feb])
pd.concat([jan, feb], ignore_index=True)

In [ ]:
pd.concat([jan, feb], keys=["January", "February"])

In [ ]:
titles = book["Title"]
titles.head()

In [ ]:
authors = book["Author"]
authors.head()

In [ ]:
pd.concat([titles, authors], axis="columns")

- The `keys` parameter creates a `MultiIndex` using the specified keys/labels.

## The pd.concat Function II
- Pandas concatenates the `DataFrame` objects along the row/index axis by default.
- Pandas will include all columns that exist in either `DataFrame`. If there are no matching values, pandas will use `NaN` values.
- Pass the `axis` parameter an argument of `"columns"` to concatenate on the column axis.

## Left Joins
- The `merge` method joins two `DataFrame` objects together based on shared values in a column or an index.
- A left join merges one `DataFrame` into another based on values in the first one.
- The "left" `DataFrame` is the one we invoke the `merge` method on.
- If the left `DataFrame`'s value is not found in the right `DataFrame`, the row will hold `NaN` values.
<img src="assets/SQL_Joins.png" width="800" height="800"/>

In [ ]:
jan.merge(book, how="left", on="Book ID")

## The left_on and right_on Parameters
- The `left_on` and `right_on` parameters designate the column names from each `DataFrame` to use in the merge.

In [ ]:
jan.merge(members, how="left", left_on="Member ID", right_on="ID").drop(columns=["ID"])

## Inner Joins I
- Inner joins merge two tables based on *shared*/*common* values in columns.
- If only one `DataFrame` has a value, pandas will exclude it from the final results set.
- If the same ID occurs multiple times, pandas will store each possible combination of the values.
- The design of the join ensures that the results will be the same no matter what `DataFrame` the `merge` method is invoked upon.
<img src="assets/SQL_Joins.png" width="800" height="800"/>

- Let's identify the members who checked out a book in both January and February.

In [ ]:
jan.merge(feb, how="inner", on="Member ID", suffixes=[" - Jan", " - Feb"])

## Matching across Multiple Columns
- We can pass a list of multiple columns to the `on` parameter of the `merge` method.
- Pandas will require matching values in both columns across the two `DataFrames`.

- Let's identify the members who checked out the _same_ in book January and February.

In [ ]:
jan.merge(feb, how="inner", on=["Member ID", "Book ID"])

## Full/Outer Join
- A **full/outer** join includes values that are found in either `DataFrame` or both `DataFrame` objects.
- Pandas does not mind if a value exists in one `DataFrame` but not the other.
- If a value does not exist in one `DataFrame`, it will have a `NaN`.

<img src="assets/SQL_Joins.png" width="800" height="800"/>

In [ ]:
merged = jan.merge(
    feb, how="outer", on="Member ID", suffixes=[" - Jan", " - Feb"], indicator=True
)
merged.head()

In [ ]:
merged["_merge"].value_counts()

In [ ]:
merged[merged["_merge"] == "both"]

## Merging by Indexes with the left_index and right_index Parameters
- Use the `on` parameter if the column(s) to be matched on have the same names in both `DataFrame` objects.
- Use the `left_on` and `right_on` parameters if the columns with matches have different names.
- Use the `left_index` or `right_index` parameters (set to `True`) if the values to be matched are found in a `DataFrame` index.

In [ ]:
books = pd.read_csv("library/books.csv", index_col="Book ID")
members = pd.read_csv(
    "library/members.csv", parse_dates=["Signup Date"], index_col="ID"
)
jan = pd.read_csv("library/checkouts_jan.csv")
feb = pd.read_csv("library/checkouts_feb.csv")

In [ ]:
jan.merge(members, how="left", left_on="Member ID", right_index=True)

## The join Method
- The `join` method is a shortcut for joining two `DataFrames` using index labels.

In [ ]:
books = pd.read_csv("library/books.csv")
members = pd.read_csv(
    "library/members.csv", parse_dates=["Signup Date"], index_col="ID"
).sort_index()
jan = pd.read_csv("library/checkouts_jan.csv", index_col="Member ID")
feb = pd.read_csv("library/checkouts_feb.csv")

In [ ]:
jan.merge(members, how="left", left_index=True, right_index=True)

In [ ]:
jan.join(members)